# Namelyze 使用教程

**学者国籍与性别推断工具 - 完整流程教学**

本教程将带你一步步了解如何使用 Namelyze 进行学者姓名的国籍和性别推断。

**联系方式:**

- **姓名:** 尚丽源
- **学校:** 复旦大学
- **邮箱:** lyshang23@m.fudan.edu.cn

## 📋 目录

1. [环境准备](#1-环境准备)
2. [配置API](#2-配置API)
3. [准备数据](#3-准备数据)
4. [运行推断](#4-运行推断)
5. [结果分析](#5-结果分析)
6. [高级用法](#6-高级用法)
7. [常见问题](#7-常见问题)

## 1. 环境准备

### 1.1 检查Python版本

In [ ]:
import sys
print(f"Python版本: {sys.version}")

# 确保Python版本 >= 3.8
assert sys.version_info >= (3, 8), "需要Python 3.8或更高版本"
print("✓ Python版本符合要求")

### 1.2 安装依赖包

In [ ]:
# 安装所需的Python包
!pip install openai pandas python-dotenv pydantic pydantic-settings tqdm tenacity -q
print("✓ 依赖包安装完成")

### 1.3 验证安装

In [ ]:
import openai
import pandas as pd
from dotenv import load_dotenv
from pydantic import __version__ as pydantic_version
import pycountry
from pathlib import Path

print("✓ 所有依赖包导入成功")
print(f"  - openai: {openai.__version__}")
print(f"  - pandas: {pd.__version__}")
print(f"  - pydantic: {pydantic_version}")
print(f"  - pycountry: {pycountry.__version__}")

## 2. 配置API

### 2.1 创建配置文件

首先，我们需要配置API信息。你可以使用任何OpenAI兼容的API服务。

根据你使用的API服务商，获取三个参数，配置示例如下：

**OpenAI**:
```bash
OPENAI_API_BASE=https://api.openai.com/v1
OPENAI_API_KEY=sk-your-key-here
MODEL_NAME=gpt-5-nano
```

**DeepSeek**:
```bash
OPENAI_API_BASE=https://api.deepseek.com
OPENAI_API_KEY=your-deepseek-key
MODEL_NAME=deepseek-chat
```

In [ ]:
OPENAI_API_BASE = "https://api.deepseek.com"
OPENAI_API_KEY = "sk-59ce710eecd844a8bdbff0957b5ac415"
MODEL_NAME =     "deepseek-chat"
BATCH_SIZE = 2
MAX_WORKERS= 2
ENABLE_CONCURRENT = True
INPUT_CSV =  "data/input/names_small.csv"
OUTPUT_CSV=  "data/output/results.csv"
NAME_COLUMN= "name"
# .env 文件内容
env_content = f"""# OpenAI Compatible API Configuration
OPENAI_API_BASE={OPENAI_API_BASE}
OPENAI_API_KEY={OPENAI_API_KEY}
MODEL_NAME={MODEL_NAME}

# Processing Configuration
BATCH_SIZE={BATCH_SIZE}
MAX_RETRIES=3
TIMEOUT=60
MAX_WORKERS={MAX_WORKERS}
ENABLE_CONCURRENT={ENABLE_CONCURRENT}

# File Paths
INPUT_CSV={INPUT_CSV}
OUTPUT_CSV={OUTPUT_CSV}
NAME_COLUMN={NAME_COLUMN}
"""

In [ ]:
env_file = Path(".env")
# 创建 .env 文件
with open(env_file, 'w', encoding='utf-8') as f:
    f.write(env_content)

### 2.2 验证配置

In [ ]:
from dotenv import load_dotenv
import os

# 加载环境变量
load_dotenv()

# 检查必需的配置项
api_key = os.getenv("OPENAI_API_KEY")
api_base = os.getenv("OPENAI_API_BASE")
model_name = os.getenv("MODEL_NAME")

print("配置检查：")
print(f"  API Base: {api_base}")
print(f"  API Key: {api_key[:10]}..." if api_key else "  API Key: 未设置 ❌")
print(f"  Model: {model_name}")

if not api_key or api_key == "sk-your-api-key-here":
    print("\n⚠️  请先在.env文件中设置有效的API密钥！")
else:
    print("\n✓ 配置验证通过")

## 3. 准备数据

In [ ]:
import pandas as pd

# 读取示例数据
sample_df = pd.read_csv(Path(INPUT_CSV))
print(f"示例数据包含 {len(sample_df)} 个学者姓名：")
print()
sample_df

## 4. 运行推断

### 4.1 导入模块

In [ ]:
import sys
sys.path.append('.')

from src.config import load_settings
from src.llm_client import LLMClient
from src.processor import ScholarProcessor

print("✓ 模块导入成功")

### 4.2 初始化系统

In [ ]:
import logging

# 配置日志
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# 加载配置
settings = load_settings()

# 初始化LLM客户端
llm_client = LLMClient(
    api_base=settings.openai_api_base,
    api_key=settings.openai_api_key,
    model_name=settings.model_name,
    timeout=settings.timeout,
    max_retries=settings.max_retries
)

# 初始化处理器
processor = ScholarProcessor(
    llm_client=llm_client,
    batch_size=settings.batch_size,
    max_workers=settings.max_workers,
    enable_concurrent=settings.enable_concurrent
)

print("✓ 系统初始化完成")

### 4.3 运行推断流程

In [ ]:
# 执行完整的处理流程
processor.run(
    input_csv=settings.input_csv,
    output_csv=settings.output_csv,
    name_column=settings.name_column
)

print("\n✓ 处理完成！")

## 5. 结果分析

### 5.1 查看结果

In [ ]:
# 读取结果文件
results_df = pd.read_csv(settings.output_csv)

print(f"处理结果共 {len(results_df)} 条记录：")
print()
results_df

### 5.2 统计分析

In [ ]:
# 统计成功和失败的数量
success_count = (results_df['has_error'] == 'No').sum()
error_count = (results_df['has_error'] == 'Yes').sum()

print(f"处理统计：")
print(f"  ✓ 成功: {success_count} ({success_count/len(results_df)*100:.1f}%)")
print(f"  ✗ 错误: {error_count} ({error_count/len(results_df)*100:.1f}%)")

# 性别分布
print(f"\n性别分布：")
gender_counts = results_df['gender'].value_counts()
for gender, count in gender_counts.items():
    print(f"  {gender}: {count}")

# 置信度分布
print(f"\n性别置信度分布：")
conf_gender_counts = results_df['conf_gender'].value_counts()
for conf, count in conf_gender_counts.items():
    print(f"  {conf}: {count}")

print(f"\n国籍置信度分布：")
conf_nation_counts = results_df['conf_nation'].value_counts()
for conf, count in conf_nation_counts.items():
    print(f"  {conf}: {count}")

### 5.3 查看错误记录

In [ ]:
# 筛选有错误的记录
error_records = results_df[results_df['has_error'] == 'Yes']

if len(error_records) > 0:
    print(f"发现 {len(error_records)} 条错误记录：")
    print()
    display(error_records[['name', 'error_reason']])
else:
    print("✓ 没有错误记录")

## 6. 高级用法

### 6.1 查看Prompt模板

In [ ]:
from src.prompt_template import generate_prompt

# 生成一个示例prompt
sample_prompt = generate_prompt(["Adam Smith", "李四"])

print("Prompt模板内容：")
print("=" * 60)
print(sample_prompt)  # 只显示前1000字符
print("...")
print("=" * 60)

## 7. 常见问题

### Q1: API调用失败怎么办？

**A:** 检查以下几点：
- API密钥是否正确
- API Base URL是否正确
- 网络连接是否正常
- 是否触发了API限流

系统会自动重试失败的请求（默认3次）。

### Q2: 为什么有些结果标记为错误？

**A:** 可能的原因：
- LLM返回的JSON格式不正确
- 某些字段缺失
- 字段值不符合预期格式

查看 `error_reason` 列可以了解具体原因。

### Q3: 如何提高准确度？

**A:** 建议：
- 使用更强大的模型（如GPT-4而非GPT-3.5）
- 对于重要数据，人工复核低置信度结果
- 考虑使用多个模型交叉验证

### Q4: 批量大小如何选择？

**A:** 建议：
- 默认值20适合大多数情况
- 如果遇到token限制，减小批量大小
- 如果模型支持更大上下文，可以增加批量大小以提高效率

### Q5: 支持哪些国家代码？

**A:** 使用ISO 3166-1 alpha-3标准（三字母代码），如：
- USA (美国)
- CHN (中国)
- GBR (英国)
- JPN (日本)
- 等等...

查看 `src/validator.py` 中的 `VALID_NATIONS` 集合获取完整列表。

## 总结

恭喜！你已经完成了Namelyze的完整教程。

### 关键要点：

1. ✅ 配置正确的API信息
2. ✅ 准备标准格式的CSV输入文件
3. ✅ 运行推断并获取结果
4. ✅ 分析结果并关注错误记录
5. ✅ 根据需要调整批量大小和模型参数

### 下一步：

- 使用自己的数据进行测试
- 根据实际需求调整配置
- 查看 README.md 了解更多详细信息
- 遇到问题查看日志文件 `namelyze.log`

祝你使用愉快！